# 07 - Detect the edited image

**Goal:** edit one image with the same controls as the slide, then run the Session 01 Lab 02 object-detection check. Change one value at a time and record what happens to the boxes and confidence.


## 1. Install and import

This notebook uses PIL for the editable image transformations and Ultralytics YOLO for the same object-detection task used in Lab 02. It uses `display()` rather than matplotlib.


In [ ]:
# Install the detector once in a new Colab runtime.
%pip -q install ultralytics

# PIL edits the image; display() shows each result without matplotlib.
from pathlib import Path
from urllib.request import urlretrieve
from PIL import Image, ImageEnhance, ImageFilter, ImageOps
from IPython.display import display
from ultralytics import YOLO


## 2. Choose the input image

Upload an image you edited yourself, or leave the fallback source in place while you learn the controls.


In [ ]:
# EDIT THIS: set the local filename you want the detector to inspect.
IMAGE_FILE = 'edited_input.jpg'
FALLBACK_URL = 'https://ultralytics.com/images/bus.jpg'

# In Colab, uncomment the next two lines to upload your edited image.
# from google.colab import files
# files.upload()

# Provide a reliable starter image only when the chosen file is not present.
image_path = Path(IMAGE_FILE)
if not image_path.exists():
    image_path = Path('bus.jpg')
    if not image_path.exists():
        urlretrieve(FALLBACK_URL, image_path)

original = Image.open(image_path).convert('RGB')
print(f'Using input: {image_path} - {original.size}')
display(original)


## 3. Edit one visual property

These values match the control ideas on the slide. Change one value, rerun the cell, and note the visible evidence that changed before you run detection.


In [ ]:
# EDIT THESE VALUES - change ONE setting at a time.
CROP_PERCENT = 0       # 0 to 45: removes equal context from each edge.
DETAIL_PERCENT = 100   # 15 to 100: lower values remove fine detail.
BLUR_RADIUS = 0        # 0 to 14: higher values soften edges.
LIGHT_FACTOR = 1.00    # 0.25 to 1.50: lower values darken the image.
COLOUR_FACTOR = 1.00   # 0.00 to 1.50: 0 removes colour.

def edit_image(image):
    # Crop first so every later operation uses the same chosen area.
    width, height = image.size
    inset = CROP_PERCENT / 200
    edited = image.crop((width * inset, height * inset, width * (1 - inset), height * (1 - inset)))

    # Reduce detail, then enlarge again so the final image keeps the same display size.
    detail_scale = max(0.15, DETAIL_PERCENT / 100)
    small = edited.resize((max(1, int(edited.width * detail_scale)), max(1, int(edited.height * detail_scale))))
    edited = small.resize((width, height))

    # Apply light, colour, and blur after the geometric changes.
    edited = ImageEnhance.Brightness(edited).enhance(LIGHT_FACTOR)
    edited = ImageEnhance.Color(edited).enhance(COLOUR_FACTOR)
    return edited.filter(ImageFilter.GaussianBlur(radius=BLUR_RADIUS))

edited = edit_image(original)
print('Edited image - record which visual clue changed before running detection.')
display(edited)


## 4. Run the Lab 02 object check

The confidence threshold is also editable. A higher threshold hides weaker boxes but can also hide a real object.


In [ ]:
# EDIT THIS: Lab 02 confidence threshold.
CONFIDENCE = 0.40  # Try 0.25, 0.40, and 0.60 one at a time.

# Load the small detection model, then run it on the edited image.
model = YOLO('yolo11n.pt')
result = model(edited, conf=CONFIDENCE, verbose=False)[0]

# result.plot() returns a BGR array, so reverse the colour channels for PIL display.
annotated_bgr = result.plot()
annotated_rgb = annotated_bgr[:, :, ::-1]
annotated = Image.fromarray(annotated_rgb)

print(f'Confidence threshold: {CONFIDENCE}')
print(f'Detections kept: {len(result.boxes)}')
display(annotated)


## 5. Record the failure point

Write one evidence statement:

- **Input:** Which edited image did you use?
- **Controls:** Which exact values did you change?
- **Output:** Which box, label, or confidence changed?
- **Failure point:** At what setting would you stop trusting the result?
- **Human check:** What should a person verify before acting on the output?
